# Esercizio: addestra il tuo mini Transformer decoder-only (Keras)
## SOLUZIONE (per il docente)

## Cosa si studia (per lo studente)
Un **Transformer decoder-only** (la stessa famiglia di architetture usata da GPT): un modello che legge una sequenza di parole e impara a prevedere, per ogni posizione, **la parola successiva** — guardando solo il testo che la precede, mai quello che segue. È la riga `use_causal_mask=True` dentro la Multi-Head Attention che rende il modello decoder-only e non encoder (vedi `05_Transformer_e_LLM_con_addestramento.ipynb`).

L'architettura (classe `MiniGPT`) è fornita già pronta: lo studente scrive solo la pipeline di addestramento.

## Obiettivo dell'esercizio (per lo studente)
Ottenere un modello che: 1) è stato addestrato per davvero (loss che scende), 2) genera nuove parole plausibili a partire da una frase seed.

## Passi richiesti (in breve)
1. Corpus + tokenizzazione + vocabolario.
2. Sequenze di input/target (next-token prediction).
3. Creazione dell'istanza `MiniGPT`.
4. `compile` e `fit`.
5. Grafico della loss.
6. Funzione di generazione testo.

## Questo file
Stessa struttura del notebook per gli studenti, con il codice della pipeline di training scritto per intero, punto per punto secondo la guida. Usalo come riferimento per la correzione: le scelte numeriche (lunghezza sequenza, dimensioni del modello, epoche...) sono solo un esempio valido, non l'unica risposta corretta.


## Setup e architettura (già pronti, non serve scriverli)

Le prossime due celle importano le librerie e definiscono la classe `MiniGPT` che userai. Eseguile e basta.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("Setup completato.")
print(f"TensorFlow version: {tf.__version__}")


In [ ]:
class PositionalEncoding(layers.Layer):
    """Aggiunge informazione sulla posizione di ciascun token."""

    def __init__(self, max_len=5000, d_model=512):
        super().__init__()
        self.d_model = d_model
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        self.pe = tf.constant(pe, dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:seq_len, :]


class DecoderBlock(layers.Layer):
    """Un blocco decoder: Multi-Head Attention causale + Feed-Forward.

    use_causal_mask=True e' cio' che rende questo blocco decoder-only
    (stile GPT): ogni token vede solo i token che lo precedono.
    """

    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate,
        )
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        attn_output = self.mha(
            x, x, x,
            training=training,
            use_causal_mask=True,
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2


class MiniGPT(tf.keras.Model):
    """Transformer decoder-only in miniatura, in stile GPT."""

    def __init__(self, vocab_size, d_model, num_heads, num_layers, dff,
                 max_len=1000, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(max_len, d_model)
        self.decoder_blocks = [
            DecoderBlock(d_model, num_heads, dff, dropout_rate)
            for _ in range(num_layers)
        ]
        self.final_layer = layers.Dense(vocab_size)
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)
        for decoder_block in self.decoder_blocks:
            x = decoder_block(x, training=training)
        return self.final_layer(x)


print("Classi definite: PositionalEncoding, DecoderBlock, MiniGPT")
print("Non devi modificarle: le userai nei passi della guida qui sotto.")


## La tua guida passo-passo

Segui questi passi **in ordine**. Non c'è codice qui: leggi cosa fare e scrivilo tu in Python nelle celle vuote più sotto (sono divise a gruppi di passi solo per comodità di esecuzione).

1. Crea una variabile chiamata `testo_train`: una stringa con **almeno 8 frasi brevi**, scritte da te. Usa poche parole in totale (8-10), che si ripetono in frasi diverse (es. "il gatto dorme", "il cane corre"...): più si ripetono, più in fretta il modello riesce a imparare con un training breve.

2. Trasforma `testo_train` in una lista di singole parole, tutte minuscole (tokenizzazione). Chiama il risultato `tokens_train`.

3. Costruisci il vocabolario: l'insieme delle parole **distinte** presenti in `tokens_train` (senza ripetizioni), ordinato. Chiamalo `vocab_train`. Crea poi due dizionari: `word_to_id` (da parola a numero) e `id_to_word` (da numero a parola). Salva anche il numero di parole distinte in `vocab_size_train`.

4. Usa `word_to_id` per trasformare `tokens_train` in una lista di numeri. Chiamala `token_ids_train`.

5. Scegli una lunghezza di sequenza — una variabile `seq_len_train` (per esempio un numero tra 3 e 6) — e costruisci due liste, `X_train` e `y_train`: scorrendo `token_ids_train`, ogni elemento di `X_train` è una finestra di `seq_len_train` id consecutivi, e il corrispondente elemento di `y_train` è la stessa finestra ma spostata di una posizione in avanti (il token che segue ciascuno di quelli in input: è il compito di "next-token prediction"). Alla fine trasforma entrambe le liste in array numpy.

6. Crea un modello `MiniGPT` (la classe è già definita sopra: non riscriverla, usala) e chiamalo `mini_gpt_train`. Usa `vocab_size_train` come vocabolario. Scegli dimensioni piccole per un training veloce: `d_model` tra 32 e 128, `num_heads` che **divida esattamente** `d_model`, `num_layers` pari a 1 o 2, `dff` circa il doppio di `d_model`, e `max_len` uguale a `seq_len_train`.

7. "Compila" `mini_gpt_train` (metodo `.compile()`): scegli un optimizer adatto a reti neurali (in classe abbiamo usato Adam), una loss adatta a un compito di classificazione con target espressi come **numeri interi** e non come vettori one-hot (in classe: `SparseCategoricalCrossentropy`, con `from_logits=True` perché il modello non applica già una softmax finale), e come metrica quella che misura la frazione di predizioni corrette.

8. Allena il modello (metodo `.fit()`), passando `X_train` e `y_train`. Scegli un numero di epoche (per esempio tra 50 e 150) e una dimensione di batch (per esempio 8). Salva il risultato di `.fit()` in una variabile chiamata `history` (serve al passo successivo).

9. Usa `history.history["loss"]` per disegnare un grafico della loss lungo le epoche (con `matplotlib.pyplot`, già importato come `plt`). Se il training ha funzionato, la loss deve scendere.

10. Scrivi una funzione che generi nuovo testo a partire da una frase seed (1-2 parole del tuo vocabolario): ad ogni passo, dai in input al modello la finestra di token più recente (lunga `seq_len_train`, riusando l'ultima parola se la frase è più corta), guarda le previsioni del modello sull'**ultima posizione**, prendi l'indice della parola con il punteggio più alto (questo si chiama "greedy decoding": in TensorFlow, la funzione che restituisce l'indice del valore massimo di un tensore è `tf.argmax`) e aggiungila alla sequenza generata. Ripeti per un numero di passi a tua scelta, poi prova la funzione con un seed che sceglierai tra le parole del tuo vocabolario.

**Come capire se hai fatto bene:** se tutti i passi sono corretti, alla fine il grafico della loss scenderà e la funzione di generazione produrrà parole che "assomigliano" al tuo corpus (non frasi a caso).


In [ ]:
# Punti 1-5 della guida: corpus, tokenizzazione, vocabolario, sequenze.

# 1. Corpus di training (almeno 8 frasi, poche parole ripetute).
testo_train = """
il gatto dorme
il gatto mangia
il gatto corre
il cane dorme
il cane mangia
il cane corre
il gatto è nero
il cane è felice
"""

# 2. Tokenizzazione: testo -> lista di parole minuscole.
tokens_train = testo_train.lower().split()

# 3. Vocabolario e dizionari di conversione parola <-> id.
vocab_train = sorted(set(tokens_train))
word_to_id = {word: i for i, word in enumerate(vocab_train)}
id_to_word = {i: word for word, i in word_to_id.items()}
vocab_size_train = len(vocab_train)

# 4. Testo tokenizzato -> lista di id numerici.
token_ids_train = [word_to_id[token] for token in tokens_train]

# 5. Sequenze di input/target per next-token prediction.
seq_len_train = 5
X_train = []
y_train = []
for i in range(len(token_ids_train) - seq_len_train):
    X_train.append(token_ids_train[i:i + seq_len_train])
    y_train.append(token_ids_train[i + 1:i + seq_len_train + 1])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"Vocabolario: {vocab_train}")
print("Input :", [id_to_word[i] for i in X_train[0]])
print("Target:", [id_to_word[i] for i in y_train[0]])
print(f"Shape Input/Target: {X_train.shape} / {y_train.shape}")


In [ ]:
# Punto 6 della guida: creazione del modello.
# num_heads=4 divide esattamente d_model=64; dff=128 e' il doppio di d_model.
mini_gpt_train = MiniGPT(
    vocab_size=vocab_size_train,
    d_model=64,
    num_heads=4,
    num_layers=2,
    dff=128,
    max_len=seq_len_train,
    dropout_rate=0.1,
)

print("Modello creato (pesi ancora casuali, training non iniziato).")


In [ ]:
# Punti 7-8 della guida: compile e fit.
mini_gpt_train.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

print("INIZIO ADDESTRAMENTO...\n")

history = mini_gpt_train.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    verbose=1,
)

print("\nADDESTRAMENTO COMPLETATO")


In [ ]:
# Punto 9 della guida: grafico della loss.
plt.figure(figsize=(9, 5))
plt.plot(history.history["loss"], linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training del Decoder-Only - Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Punto 10 della guida: generazione autoregressiva (greedy decoding).
def genera_testo(modello, seed_words, n_token_da_generare=5, seq_len=seq_len_train):
    token_ids = [word_to_id[w] for w in seed_words]

    for _ in range(n_token_da_generare):
        contesto = token_ids[-seq_len:]
        while len(contesto) < seq_len:
            contesto = [contesto[0]] + contesto

        input_ids = tf.constant([contesto])
        logits = modello(input_ids, training=False)

        prossimo_logit = logits[0, -1, :]
        prossimo_id = int(tf.argmax(prossimo_logit).numpy())

        token_ids.append(prossimo_id)

    return [id_to_word[i] for i in token_ids]


seed = ["il", "gatto"]
generato = genera_testo(mini_gpt_train, seed, n_token_da_generare=5)

print(f"Seed: {' '.join(seed)}")
print(f"Generato: {' '.join(generato)}")


## Domanda finale — risposta attesa

Con poche epoche, la loss non avrebbe avuto tempo di scendere e il testo generato sarebbe stato quasi casuale. Con un corpus meno ripetitivo (frasi tutte diverse, parole usate una sola volta), il modello avrebbe avuto molti meno esempi da cui imparare gli stessi pattern, e la generazione sarebbe risultata meno coerente: il modello impara ripetendo, non "capendo" il significato delle parole.
